# UC04 — TCN và Transformer

Chạy trên một T4 nếu có; notebook hiển thị thiết bị thực tế. Hai mô hình dùng cùng split/nhãn và evaluator. Kết quả là pilot thăm dò, không phải final test.

In [ ]:
from pathlib import Path
import sys
import os

# Sửa SOURCE_ROOT nếu dùng source được gắn qua Kaggle Input.
SOURCE_ROOT = Path.cwd()
if not (SOURCE_ROOT / "src").is_dir() and (SOURCE_ROOT.parent / "src").is_dir():
    SOURCE_ROOT = SOURCE_ROOT.parent
# SOURCE_ROOT = Path("/kaggle/input/safeanes-source")
assert (SOURCE_ROOT / "src" / "safeanes").is_dir(), "Đặt SOURCE_ROOT tới repository"
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else SOURCE_ROOT
sys.path.insert(0, str(SOURCE_ROOT / "src"))
# Không đưa .local_deps Windows sang Kaggle.
if os.name == "nt" and (SOURCE_ROOT / ".local_deps").is_dir():
    sys.path.insert(0, str(SOURCE_ROOT / ".local_deps"))
print("Source:", SOURCE_ROOT, "Output:", WORK_ROOT)


In [ ]:
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import torch
print("PyTorch:", torch.__version__)
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — không phải benchmark T4")
# Trên Kaggle dùng PyTorch có CUDA sẵn trong môi trường, không cài wheel CPU.
DATASET = WORK_ROOT / "data/pilot_v1"
SEQUENCES = WORK_ROOT / "data/sequences_v1"
# Nếu dùng Input từ notebook CPU, sửa hai đường dẫn trên.
RUN_ROOT = WORK_ROOT / "artifacts"


In [ ]:
import json
from dataclasses import asdict
from safeanes.training import TrainConfig, train_sequence
from safeanes.sequences import load_dataset

reports = {}
dataset_meta, _, _, _ = load_dataset(DATASET)
for architecture in ("tcn", "transformer"):
    config = TrainConfig(**json.loads((SOURCE_ROOT / "configs" / f"{architecture}.json").read_text()))
    output = RUN_ROOT / f"{architecture}_v1"
    if (output / "report.json").is_file():
        saved_config = json.loads((output / "config.json").read_text())
        saved_env = json.loads((output / "environment.json").read_text())
        assert saved_config == asdict(config), "Config đã đổi; chọn tên run mới"
        assert saved_env["dataset_hash"] == dataset_meta["windows_sha256"], "Dataset đã đổi; chọn tên run mới"
        reports[architecture] = json.loads((output / "report.json").read_text())
        print("Đọc lại run hoàn tất:", output)
        continue
    resume = (output / "last.pt").is_file()
    reports[architecture] = train_sequence(DATASET, SEQUENCES, output, config, repeats=200, resume=resume)


In [ ]:
import pandas as pd
rows = []
for architecture, report in reports.items():
    for model, result in report["models"].items():
        rows.append({"model": model, **result["pilot_test"],
                     "all_targets_met": result["gates"]["all_point_targets_met"]})
display(pd.DataFrame(rows)[["model", "auroc", "average_precision", "event_sensitivity",
                          "alarm_ppv", "false_alarms_per_hour", "ece", "all_targets_met"]])


Đọc `history.json`, `environment.json`, `report.json` trước diễn giải. Chọn ứng viên bằng validation; ba seed/ablation tạo run mới theo runbook. Không chọn seed theo pilot_test. Benchmark T4 cần lưu đúng tên GPU và peak VRAM thực đo.